# PhishGuard: 04. Unseen-Domain Generalization Evaluation

## Academic Context & Core Research Question
> **Research Question:** Can machine-learning models identify previously unseen phishing URLs using lexical and structural characteristics of URLs rather than relying only on known malicious blacklists?

### Scope & Positioning Disclosure
- **Formal Experiment Name:** *Unseen-Domain Grouped Evaluation*
- **Security Context:** *Zero-day-like generalization setting / proxy evaluation for previously unseen phishing domains.*
- **Important Caveat:** This offline experiment evaluates out-of-distribution domain generalization under strict domain grouping. It does **not** claim guaranteed detection of all real-world zero-day phishing campaigns.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_extractor import FEATURE_NAMES
from src.preprocessing import get_or_create_processed_dataset
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

sns.set_theme(style="whitegrid")
print("Environment ready.")

## 1. Domain-Grouped Splitting (Zero Overlap Audit)
We use `StratifiedGroupKFold` to enforce that all URLs from any single domain belong exclusively to either train or test.

In [ ]:
df = get_or_create_processed_dataset()
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(sgkf.split(df, df["target"], groups=df["Domain"]))

train_domains = set(df.loc[train_idx, "Domain"])
test_domains = set(df.loc[test_idx, "Domain"])
overlap = train_domains.intersection(test_domains)

print(f"Total Unique Domains: {df['Domain'].nunique():,}")
print(f"Training Domains:     {len(train_domains):,} ({len(train_idx):,} URLs)")
print(f"Testing Domains:      {len(test_domains):,} ({len(test_idx):,} URLs)")
print(f"Domain Overlap Count: {len(overlap)} (Status: {'PASS' if len(overlap) == 0 else 'FAIL'})")

## 2. Load Evaluation Metrics & Generalization Gap
We load the comparative metrics generated during Phase 6.

In [ ]:
metrics_csv = PROJECT_ROOT / "reports" / "results" / "phase6_unseen_domain_metrics.csv"
df_comp = pd.read_csv(metrics_csv)
df_comp

## 3. Visualizing Generalization Performance

In [ ]:
df_melt = pd.melt(df_comp, id_vars=["Model", "Evaluation_Setting"], value_vars=["Accuracy", "Recall", "F1-Score", "ROC-AUC"], var_name="Metric", value_name="Score")
plt.figure(figsize=(10, 5), dpi=150)
sns.barplot(data=df_melt, x="Metric", y="Score", hue="Evaluation_Setting", palette=["#4dabf7", "#ff922b"])
plt.title("Generalization: Standard Random Split vs. Unseen-Domain Grouped Split", fontweight="bold")
plt.ylim(0.98, 1.002)
plt.legend(loc="lower right")
plt.show()

## 4. Unseen-Domain False-Negative Profile
Why do models miss certain phishing links even when trained on 176,000 domains?

In [ ]:
import json
json_path = PROJECT_ROOT / "reports" / "results" / "phase6_unseen_domain_evaluation.json"
with open(json_path) as f:
    res_json = json.load(f)

rf_fn = res_json["false_negative_structural_profile"]["RandomForestClassifier"]
print(f"Total Missed Phishing Links: {rf_fn['unseen_false_negatives_count']} / 20,104")
print(f"  - HTTPS Enabled:           {rf_fn['pct_with_https']}%")
print(f"  - No Suspicious Keywords:  {rf_fn['pct_with_no_keywords']}%")
print(f"  - Zero Subdomains:         {rf_fn['pct_with_zero_subdomains']}%")
print(f"  - Mean Path Length:        {rf_fn['structural_averages']['mean_path_length']} chars")